# 1. Data

Goal: one labelled dataset of emails.

- **1 = prompt injection**, from [LLMail-Inject](https://huggingface.co/datasets/microsoft/llmail-inject-challenge) (Microsoft, MIT licence).
- **0 = normal email**: LLMail texts the challenge marked as no attack, the challenge's own legitimate emails, and [Enron](https://huggingface.co/datasets/corbt/enron-emails) emails.

Output: `data/processed/emails.parquet`, 30 000 emails per class, fixed train/test split.

In [1]:
import json
import os
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from huggingface_hub import hf_hub_download
from pyspark.sql import functions as F

from src.spark_utils import get_spark

RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed" / "emails.parquet"
SEED = 42
PER_CLASS = 30_000

spark = get_spark()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/25 12:43:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Download

Pinned revisions: the data cannot change under us.

In [2]:
LLMAIL = ("microsoft/llmail-inject-challenge", "1063bdf01ec8762b812d5e06ee768a06faa5a6f7")
ENRON = ("corbt/enron-emails", "cfc06c758093d90993abce1a43668fb7357258a6")

llmail_files = [
    "labelled_unique_submissions_phase1.json",
    "labelled_unique_submissions_phase2.json",
    "emails_for_fp_tests.json",
    "scenarios.json",
]
for name in llmail_files:
    hf_hub_download(LLMAIL[0], f"data/{name}", repo_type="dataset", revision=LLMAIL[1], local_dir=RAW / "llmail")
for i in range(3):
    hf_hub_download(ENRON[0], f"data/train-0000{i}-of-00003.parquet", repo_type="dataset", revision=ENRON[1], local_dir=RAW / "enron")

## LLMail-Inject

Each phase is one big JSON object, `text -> label`, not one record per line. Pandas reads it. Spark takes over after.

`attack_attempt` is `true`, `false` or `unclear`, stored sometimes as text, sometimes as boolean. We keep `true` and `false`. `unclear` is dropped: the challenge's own judge could not decide.

`reason` says who set the label: `api_triggered` (the attack made the LLM call the email API, so the label is sure) or `judge` (the challenge's LLM judge). Needed to compare ML and LLMs fairly: an LLM judged against labels from another LLM has an edge.

In [3]:
rows = []
for phase in (1, 2):
    path = RAW / "llmail" / "data" / f"labelled_unique_submissions_phase{phase}.json"
    for text, info in json.loads(path.read_text()).items():
        value = str(info["attack_attempt"]).lower()
        if value in ("true", "false"):
            rows.append((text, int(value == "true"), "llmail", phase, info.get("judge_category"), info["reason"]))

llmail = pd.DataFrame(rows, columns=["text", "label", "source", "phase", "technique", "reason"])
llmail.groupby(["phase", "label"]).size().unstack()

label,0,1
phase,,
1,9452,127376
2,2500,21007


## Legitimate emails from the challenge

Benign emails the challenge used itself. Same style as the attacks: good negatives.

In [4]:
data_dir = RAW / "llmail" / "data"
fp_tests = json.loads((data_dir / "emails_for_fp_tests.json").read_text())
scenarios = json.loads((data_dir / "scenarios.json").read_text())
benign = fp_tests + [email for s in scenarios.values() for email in s["emails"]]

challenge = pd.DataFrame({"text": benign, "label": 0, "source": "challenge", "phase": None, "technique": None, "reason": None})
print(len(challenge), "emails,", challenge["text"].nunique(), "distinct")

282 emails, 238 distinct


## Enron

517 401 real emails, already split into subject and body. Big Parquet files: a job for Spark.

Forwarded and quoted parts are older emails, so we keep only what the sender wrote. Then we use the same template as LLMail, `Subject of the email: ... Body: ...`. Without it, the template alone would give away the source.

In [5]:
enron_raw = spark.read.parquet(str(RAW / "enron" / "data"))
own_text = F.trim(F.split("body", r"(?i)-{5,}\s*(original message|forwarded by)").getItem(0))

enron = (
    enron_raw.select(F.coalesce("subject", F.lit("")).alias("subject"), own_text.alias("body"))
    .filter(F.length("body") > 0)
    .select(
        F.concat(F.lit("Subject of the email: "), "subject", F.lit(".   Body: "), "body").alias("text"),
        F.lit(0).alias("label"),
        F.lit("enron").alias("source"),
        F.lit(None).cast("int").alias("phase"),
        F.lit(None).cast("string").alias("technique"),
        F.lit(None).cast("string").alias("reason"),
    )
)
print(enron_raw.count(), "Enron emails,", enron.count(), "with own text")

517401 Enron emails, 484384 with own text


## Duplicates

LLMail has many near-copies: same attack, another number or more spaces. Enron has the same email in several folders. If copies end up in both train and test, scores are too good.

Rule: same text up to case, digits and spaces = duplicate. The key is hashed: shorter, same result. A few texts carry both labels: we drop them.

In [6]:
small = pd.concat([llmail, challenge], ignore_index=True)
small["phase"] = small["phase"].astype("Int64")
schema = "text string, label int, source string, phase int, technique string, reason string"
emails = spark.createDataFrame(small, schema=schema).unionByName(enron)

normalized = F.trim(F.regexp_replace(F.regexp_replace(F.lower("text"), r"\d", "0"), r"\s+", " "))
emails = emails.withColumn("key", F.sha2(normalized, 256)).dropDuplicates(["key", "label"])
both_labels = emails.groupBy("key").count().filter("count > 1")
emails = emails.join(both_labels, "key", "left_anti")

emails.groupBy("source").pivot("label").count().orderBy("source").toPandas()

,source,0,1
0,challenge,237,NaN
1,enron,225806,NaN
2,llmail,11511,141737.0


## Balance and split

30 000 emails per class: enough to learn, small enough for a laptop. All LLMail negatives are kept, since they share the style of the attacks. Enron fills the rest.

Then 80 % train, 20 % test. Fixed once and saved, so every model sees the same split.

In [7]:
positives = emails.filter("label = 1").orderBy(F.rand(SEED)).limit(PER_CLASS)
llmail_negatives = emails.filter("label = 0 and source != 'enron'")
n_enron = PER_CLASS - llmail_negatives.count()
enron_negatives = emails.filter("source = 'enron'").orderBy(F.rand(SEED)).limit(n_enron)

dataset = (
    positives.unionByName(llmail_negatives).unionByName(enron_negatives)
    .withColumn("split", F.when(F.rand(SEED) < 0.8, "train").otherwise("test"))
    .drop("key")
)
dataset.write.mode("overwrite").parquet(str(OUT))

## Check

Read back from disk: what the next notebooks will use.

In [8]:
saved = spark.read.parquet(str(OUT))
print(saved.count(), "emails")
saved.groupBy("source", "label").pivot("split").count().orderBy("label", "source").toPandas()

60000 emails


,source,label,test,train
0,challenge,0,43,194
1,enron,0,3684,14568
2,llmail,0,2268,9243
3,llmail,1,5968,24032


In [9]:
# Phase of the injections: needed for the phase 1 -> phase 2 test.
saved.filter("label = 1").groupBy("phase").pivot("split").count().orderBy("phase").toPandas()

,phase,test,train
0,1,5116,20645
1,2,852,3387


In [10]:
# Who set the label of the injections: needed for the ML vs LLM comparison.
saved.filter("label = 1").groupBy("reason").pivot("split").count().orderBy("reason").toPandas()

,reason,test,train
0,api_triggered,972,4030
1,judge,4996,20002
